In [ ]:
# CELL 1 — GPU + Mixed Precision
import tensorflow as tf

print("GPUs:", tf.config.list_physical_devices("GPU"))
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

In [ ]:
# CELL 2 — Paths + Run Config

import os
import kagglehub

# Download dataset
dataset_root = kagglehub.dataset_download("chat4uchat/mimc-bp")

if not os.path.exists(dataset_root):
    raise FileNotFoundError(f"Dataset not found at: {dataset_root}")

print("Downloaded root:", dataset_root)
print("Top-level files:", os.listdir(dataset_root))

# Actual dataset folder
dataset_root = os.path.join(dataset_root, "mimic_bp")

if not os.path.exists(dataset_root):
    raise FileNotFoundError(f"Inner dataset folder not found at: {dataset_root}")

print("Actual dataset root:", dataset_root)
print("Dataset contents:", os.listdir(dataset_root))

# Work directory
work_dir = "/kaggle/working/mimic_bp"
os.makedirs(work_dir, exist_ok=True)

# Modalities Config
MOD1 = "ppg"
MOD2 = "resp"
MOD3 = "resp"

RUN_ID = 1

assert MOD1 != MOD2, "MOD1 and MOD2 must be different"

# Model Type
MODEL_TYPE = "cnn_bilstm_aux"

# Run Name
run_name = f"{MOD1.upper()}_{MOD2.upper()}_{MODEL_TYPE}_RUN{RUN_ID}"

# Checkpoint directory
ckpt_dir = os.path.join("/kaggle/working", "dual_modality_runs", run_name)
os.makedirs(ckpt_dir, exist_ok=True)

print("run_name:", run_name)
print("ckpt_dir:", ckpt_dir)
print("work_dir:", work_dir)

In [ ]:
# CELL 3 — Use dataset directly

import os

files = os.listdir(dataset_root)
print("Available files:", files)

required_dirs = ["abp", "ecg", "ppg", "resp", "labels"]
missing = [name for name in required_dirs if not os.path.exists(os.path.join(dataset_root, name))]

if missing:
    raise FileNotFoundError(f"Missing required folders: {missing}")

base_dir = dataset_root
print("Base directory:", base_dir)

In [ ]:
# CELL 4 — Load splits + overlap check + inspect one subject

import ast
import os
import numpy as np

def load_ids_fixed(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing split file: {path}")
    with open(path, "r") as f:
        t = f.read().strip()

    if t.startswith("[") and t.endswith("]"):
        return ast.literal_eval(t)
    return [line.strip() for line in t.splitlines() if line.strip()]

# Load splits from Kaggle dataset root
train_ids = load_ids_fixed(os.path.join(dataset_root, "train_subjects.txt"))
val_ids   = load_ids_fixed(os.path.join(dataset_root, "val_subjects.txt"))
test_ids  = load_ids_fixed(os.path.join(dataset_root, "test_subjects.txt"))

print("Counts:", {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)})

# Overlap check
overlap_tv = set(train_ids) & set(val_ids)
overlap_tt = set(train_ids) & set(test_ids)
overlap_vt = set(val_ids) & set(test_ids)

print("Overlap train/val :", len(overlap_tv))
print("Overlap train/test:", len(overlap_tt))
print("Overlap val/test  :", len(overlap_vt))

if overlap_tv or overlap_tt or overlap_vt:
    raise ValueError("Data leakage detected between splits!")

if len(train_ids) == 0:
    raise ValueError("train_ids is empty. Check split files.")

# Inspect one sample
sid = train_ids[0]
print("Sample train id:", sid)

p1 = os.path.join(base_dir, MOD1, f"{sid}_{MOD1}.npy")
p2 = os.path.join(base_dir, MOD2, f"{sid}_{MOD2}.npy")
p3 = os.path.join(base_dir, MOD3, f"{sid}_{MOD3}.npy")
lp = os.path.join(base_dir, "labels", f"{sid}_labels.npy")

print("Signal 1:", p1, "| exists?", os.path.exists(p1))
print("Signal 2:", p2, "| exists?", os.path.exists(p2))
print("Signal 3:", p3, "| exists?", os.path.exists(p3))
print("Labels  :", lp, "| exists?", os.path.exists(lp))

if not (os.path.exists(p1) and os.path.exists(p2) and os.path.exists(lp)):
    raise FileNotFoundError("Missing signal/label file. Check dataset paths or IDs.")

# Load arrays
x1 = np.load(p1)
x2 = np.load(p2)
y  = np.load(lp)

print("sig1 shape:", x1.shape, "| dtype:", x1.dtype)
print("sig2 shape:", x2.shape, "| dtype:", x2.dtype)
print("labels shape:", y.shape, "| dtype:", y.dtype)

# Sanity checks
assert x1.shape == x2.shape, "Modalities not aligned!"
assert x1.shape[0] == y.shape[0], "Signal-label mismatch!"

print("Segment length L =", x1.shape[-1])

In [ ]:
# CELL 5 — Dataset Generator

import numpy as np
import tensorflow as tf
import os

WINDOW_LEN = 3750
BATCH = 64

def gen_tri_full_segment(subject_ids, mod1, mod2, mod3):
    sig1_dir = os.path.join(base_dir, mod1)
    sig2_dir = os.path.join(base_dir, mod2)
    sig3_dir = os.path.join(base_dir, mod3)
    labels_dir = os.path.join(base_dir, "labels")

    missing = 0
    total_samples = 0

    for sid in subject_ids:
        p1 = os.path.join(sig1_dir, f"{sid}_{mod1}.npy")
        p2 = os.path.join(sig2_dir, f"{sid}_{mod2}.npy")
        p3 = os.path.join(sig3_dir, f"{sid}_{mod3}.npy")
        lp = os.path.join(labels_dir, f"{sid}_labels.npy")

        if not (os.path.exists(p1) and os.path.exists(p2) and os.path.exists(p3) and os.path.exists(lp)):
            missing += 1
            continue

        s1 = np.load(p1)   # (num_segments, 3750)
        s2 = np.load(p2)   # (num_segments, 3750)
        s3 = np.load(p3)   # (num_segments, 3750)
        y  = np.load(lp)   # (num_segments, 2)

        assert s1.shape == s2.shape == s3.shape, f"Shape mismatch for subject {sid}"
        assert s1.shape[0] == y.shape[0], f"Label mismatch for subject {sid}"

        for i in range(s1.shape[0]):
            x1 = s1[i].astype("float32").reshape(WINDOW_LEN, 1)
            x2 = s2[i].astype("float32").reshape(WINDOW_LEN, 1)
            x3 = s3[i].astype("float32").reshape(WINDOW_LEN, 1)
            yy = y[i].astype("float32")

            if (
                np.isfinite(x1).all()
                and np.isfinite(x2).all()
                and np.isfinite(x3).all()
                and np.isfinite(yy).all()
            ):
                total_samples += 1
                yield (x1, x2, x3), yy

    if missing > 0:
        print(f"Missing files for {missing} subjects in this split.")
    print(f"Total valid samples: {total_samples}")


output_signature = (
    (
        tf.TensorSpec(shape=(WINDOW_LEN, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(WINDOW_LEN, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(WINDOW_LEN, 1), dtype=tf.float32),
    ),
    tf.TensorSpec(shape=(2,), dtype=tf.float32),
)

train_ds_raw = tf.data.Dataset.from_generator(
    lambda: gen_tri_full_segment(train_ids, MOD1, MOD2, MOD3),
    output_signature=output_signature
)

val_ds_raw = tf.data.Dataset.from_generator(
    lambda: gen_tri_full_segment(val_ids, MOD1, MOD2, MOD3),
    output_signature=output_signature
)

test_ds_raw = tf.data.Dataset.from_generator(
    lambda: gen_tri_full_segment(test_ids, MOD1, MOD2, MOD3),
    output_signature=output_signature
)

print("Datasets ready (tri-modal full 30s segments)")
print("WINDOW_LEN:", WINDOW_LEN, "| channels per modality:", 1, "| BATCH:", BATCH)

# shape sanity check
(x1_0, x2_0, x3_0), y0 = next(iter(train_ds_raw.take(1)))
print("one sample x1:", x1_0.shape)   # (3750, 1)
print("one sample x2:", x2_0.shape)   # (3750, 1)
print("one sample x3:", x3_0.shape)   # (3750, 1)
print("one sample y :", y0.shape)     # (2,)

In [ ]:
# CELL 6 — Count valid multimodal samples

import os
import numpy as np

def count_tri_segments(subject_ids, mod1, mod2, mod3):
    sig1_dir = os.path.join(base_dir, mod1)
    sig2_dir = os.path.join(base_dir, mod2)
    sig3_dir = os.path.join(base_dir, mod3)
    labels_dir = os.path.join(base_dir, "labels")

    found, missing, total_segments = 0, 0, 0

    for sid in subject_ids:
        p1 = os.path.join(sig1_dir, f"{sid}_{mod1}.npy")
        p2 = os.path.join(sig2_dir, f"{sid}_{mod2}.npy")
        p3 = os.path.join(sig3_dir, f"{sid}_{mod3}.npy")
        lp = os.path.join(labels_dir, f"{sid}_labels.npy")

        if not (os.path.exists(p1) and os.path.exists(p2) and os.path.exists(p3) and os.path.exists(lp)):
            missing += 1
            continue

        s1 = np.load(p1)
        s2 = np.load(p2)
        s3 = np.load(p3)
        y  = np.load(lp)

        if not (s1.shape[0] == s2.shape[0] == s3.shape[0] == y.shape[0]):
            missing += 1
            continue

        found += 1
        total_segments += s1.shape[0]

    return found, missing, total_segments


tr_found, tr_miss, tr_n = count_tri_segments(train_ids, MOD1, MOD2, MOD3)
va_found, va_miss, va_n = count_tri_segments(val_ids, MOD1, MOD2, MOD3)
te_found, te_miss, te_n = count_tri_segments(test_ids, MOD1, MOD2, MOD3)

print("=== Dataset sample counts (tri full 30s segments) ===")
print(f"Train: subjects={tr_found} missing={tr_miss} samples={tr_n}")
print(f"Val  : subjects={va_found} missing={va_miss} samples={va_n}")
print(f"Test : subjects={te_found} missing={te_miss} samples={te_n}")

STEPS_PER_EPOCH = max(1, tr_n // BATCH)
VAL_STEPS       = max(1, va_n // BATCH)

print("STEPS_PER_EPOCH:", STEPS_PER_EPOCH)
print("VAL_STEPS      :", VAL_STEPS)

In [ ]:
# CELL 7 — Dataset statistics

import pandas as pd
import numpy as np
import os

def load_all_labels(subject_ids):
    labels_dir = os.path.join(base_dir, "labels")
    all_y = []
    miss = 0

    for sid in subject_ids:
        lp = os.path.join(labels_dir, f"{sid}_labels.npy")

        if not os.path.exists(lp):
            miss += 1
            continue

        y = np.load(lp).astype("float32")  # (num_segments, 2)

        if not np.isfinite(y).all():
            continue

        all_y.append(y)

    if not all_y:
        raise ValueError(" No labels loaded.")

    return np.concatenate(all_y, axis=0), miss


# Load splits
y_train_all, miss_train = load_all_labels(train_ids)
y_val_all,   miss_val   = load_all_labels(val_ids)
y_test_all,  miss_test  = load_all_labels(test_ids)


def stats_1d(a):
    return {
        "min": float(a.min()),
        "max": float(a.max()),
        "mean": float(a.mean()),
        "std": float(a.std()),
    }


rows = []

for name, y in [("Train", y_train_all), ("Val", y_val_all), ("Test", y_test_all)]:
    sbp_stats = stats_1d(y[:, 0])
    dbp_stats = stats_1d(y[:, 1])

    rows.append({
        "Split": name,
        "Samples": int(y.shape[0]),
        "SBP_min": sbp_stats["min"],
        "SBP_max": sbp_stats["max"],
        "SBP_mean": sbp_stats["mean"],
        "SBP_std": sbp_stats["std"],
        "DBP_min": dbp_stats["min"],
        "DBP_max": dbp_stats["max"],
        "DBP_mean": dbp_stats["mean"],
        "DBP_std": dbp_stats["std"],
    })

df_stats = pd.DataFrame(rows)
print(df_stats)


dataset_stats = {
    "missing_label_subjects": {
        "train": int(miss_train),
        "val": int(miss_val),
        "test": int(miss_test),
    },
    "table": df_stats.to_dict(orient="records"),
}

In [ ]:
# CELL 8 — Train-only Normalization (FINAL with repeat)

import numpy as np
import tensorflow as tf


# Estimate mean/std
def estimate_mean_std(ds_raw, num_batches=200):
    s1 = s2 = c1 = 0.0
    s1_2 = s2_2 = c2 = 0.0
    s1_3 = s2_3 = c3 = 0.0

    for (x1, x2, x3), _ in ds_raw.batch(BATCH).take(num_batches):
        a1 = x1.numpy().reshape(-1)
        a2 = x2.numpy().reshape(-1)
        a3 = x3.numpy().reshape(-1)

        # MOD1
        s1 += a1.sum()
        s2 += (a1 * a1).sum()
        c1 += a1.shape[0]

        # MOD2
        s1_2 += a2.sum()
        s2_2 += (a2 * a2).sum()
        c2 += a2.shape[0]

        # MOD3
        s1_3 += a3.sum()
        s2_3 += (a3 * a3).sum()
        c3 += a3.shape[0]

    # compute stats
    mean1 = s1 / c1
    std1  = np.sqrt((s2 / c1) - mean1**2) + 1e-8

    mean2 = s1_2 / c2
    std2  = np.sqrt((s2_2 / c2) - mean2**2) + 1e-8

    mean3 = s1_3 / c3
    std3  = np.sqrt((s2_3 / c3) - mean3**2) + 1e-8

    return float(mean1), float(std1), float(mean2), float(std2), float(mean3), float(std3)


mean1, std1, mean2, std2, mean3, std3 = estimate_mean_std(train_ds_raw)

print("Z-score stats:")
print("MOD1:", MOD1, "mean =", mean1, "std =", std1)
print("MOD2:", MOD2, "mean =", mean2, "std =", std2)
print("MOD3:", MOD3, "mean =", mean3, "std =", std3)


# Convert to TF constants
mean1_tf = tf.constant(mean1, dtype=tf.float32)
std1_tf  = tf.constant(std1,  dtype=tf.float32)

mean2_tf = tf.constant(mean2, dtype=tf.float32)
std2_tf  = tf.constant(std2,  dtype=tf.float32)

mean3_tf = tf.constant(mean3, dtype=tf.float32)
std3_tf  = tf.constant(std3,  dtype=tf.float32)


# Normalize function
def map_fn(x, y):
    x1, x2, x3 = x
    x1 = (x1 - mean1_tf) / std1_tf
    x2 = (x2 - mean2_tf) / std2_tf
    x3 = (x3 - mean3_tf) / std3_tf
    return (x1, x2, x3), y


# Build datasets
train_data = (
    train_ds_raw
    .shuffle(8000)
    .batch(BATCH)
    .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
    .repeat()
)

val_data = (
    val_ds_raw
    .batch(BATCH)
    .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
    .repeat()
)

test_data = (
    test_ds_raw
    .batch(BATCH)
    .map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)


# Sanity check
(x1b, x2b, x3b), yb = next(iter(train_data.take(1)))

print("x1 batch:", x1b.shape)  # (B, 3750, 1)
print("x2 batch:", x2b.shape)  # (B, 3750, 1)
print("x3 batch:", x3b.shape)  # (B, 3750, 1)
print("y batch :", yb.shape)   # (B, 2)

In [ ]:
# CELL — Count tri samples for training steps

import os
import numpy as np

def count_tri_segments(subject_ids, mod1, mod2, mod3):
    sig1_dir = os.path.join(base_dir, mod1)
    sig2_dir = os.path.join(base_dir, mod2)
    sig3_dir = os.path.join(base_dir, mod3)
    labels_dir = os.path.join(base_dir, "labels")

    found = 0
    missing = 0
    total_segments = 0

    for sid in subject_ids:
        p1 = os.path.join(sig1_dir, f"{sid}_{mod1}.npy")
        p2 = os.path.join(sig2_dir, f"{sid}_{mod2}.npy")
        p3 = os.path.join(sig3_dir, f"{sid}_{mod3}.npy")
        lp = os.path.join(labels_dir, f"{sid}_labels.npy")

        if not (os.path.exists(p1) and os.path.exists(p2) and os.path.exists(p3) and os.path.exists(lp)):
            missing += 1
            continue

        s1 = np.load(p1)
        s2 = np.load(p2)
        s3 = np.load(p3)
        y  = np.load(lp)

        if not (s1.shape[0] == s2.shape[0] == s3.shape[0] == y.shape[0]):
            missing += 1
            continue

        found += 1
        total_segments += s1.shape[0]

    return found, missing, total_segments


tr_found, tr_miss, tr_n = count_tri_segments(train_ids, MOD1, MOD2, MOD3)
va_found, va_miss, va_n = count_tri_segments(val_ids, MOD1, MOD2, MOD3)
te_found, te_miss, te_n = count_tri_segments(test_ids, MOD1, MOD2, MOD3)

print("=== Dataset sample counts (tri full 30s segments) ===")
print(f"Train: subjects={tr_found} missing={tr_miss} samples={tr_n}")
print(f"Val  : subjects={va_found} missing={va_miss} samples={va_n}")
print(f"Test : subjects={te_found} missing={te_miss} samples={te_n}")

STEPS_PER_EPOCH = max(1, tr_n // BATCH)
VAL_STEPS       = max(1, va_n // BATCH)

print("STEPS_PER_EPOCH:", STEPS_PER_EPOCH)
print("VAL_STEPS      :", VAL_STEPS)

In [ ]:
# CELL 9 — Baseline

import numpy as np

# Collect true labels from test set
y_true = []
for _, yb in test_data:
    y_true.append(yb.numpy())
y_true = np.concatenate(y_true, axis=0)

# Train-set mean baseline
train_mean_sbp = float(np.mean(y_train_all[:, 0]))
train_mean_dbp = float(np.mean(y_train_all[:, 1]))

baseline_pred = np.zeros_like(y_true, dtype=np.float32)
baseline_pred[:, 0] = train_mean_sbp
baseline_pred[:, 1] = train_mean_dbp

baseline = {
    "train_mean_sbp": train_mean_sbp,
    "train_mean_dbp": train_mean_dbp,
    "mae_sbp": float(np.mean(np.abs(baseline_pred[:, 0] - y_true[:, 0]))),
    "mae_dbp": float(np.mean(np.abs(baseline_pred[:, 1] - y_true[:, 1]))),
    "mae_all": float(np.mean(np.abs(baseline_pred - y_true))),
    "mse_sbp": float(np.mean((baseline_pred[:, 0] - y_true[:, 0]) ** 2)),
    "mse_dbp": float(np.mean((baseline_pred[:, 1] - y_true[:, 1]) ** 2)),
    "mse_all": float(np.mean((baseline_pred - y_true) ** 2)),
}

print("=== Baseline (Train Mean) ===")
print("Train mean SBP:", baseline["train_mean_sbp"], "| Train mean DBP:", baseline["train_mean_dbp"])
print("MAE ALL:", baseline["mae_all"])
print("MAE SBP:", baseline["mae_sbp"])
print("MAE DBP:", baseline["mae_dbp"])
print("MSE ALL:", baseline["mse_all"])
print("MSE SBP:", baseline["mse_sbp"])
print("MSE DBP:", baseline["mse_dbp"])

In [ ]:
# CELL 10 — Callbacks

import os
import time
import tensorflow as tf

EPOCHS = 100
SAVE_EVERY_N = 10

best_path = os.path.join(ckpt_dir, f"best_{run_name}.keras")

# TensorBoard directory
tb_log_dir = os.path.join(ckpt_dir, "tensorboard_logs")

# CSV logger file
csv_log_path = os.path.join(ckpt_dir, f"training_log_{run_name}.csv")


class TimeInLogs(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            logs["time_sec"] = time.time() - self.t0


class SaveEveryN(tf.keras.callbacks.Callback):
    def __init__(self, n=10):
        super().__init__()
        self.n = n

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.n == 0:
            path = os.path.join(
                ckpt_dir,
                f"checkpoint_{run_name}_epoch_{epoch+1}.keras"
            )
            self.model.save(path)
            print(f"\nSaved checkpoint: {path}")


callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=best_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.TensorBoard(
        log_dir=tb_log_dir,
        histogram_freq=1,
        write_graph=True,
        write_images=False,
        update_freq="epoch"
    ),

    tf.keras.callbacks.CSVLogger(
        filename=csv_log_path,
        separator=",",
        append=False
    ),

    TimeInLogs(),
    SaveEveryN(n=SAVE_EVERY_N),
]

print("best_path   :", best_path)
print("tb_log_dir  :", tb_log_dir)
print("csv_log_path:", csv_log_path)

In [ ]:
# CELL — Build Multimodal Model

import tensorflow as tf
from tensorflow.keras import layers, Model

# Metrics
def mae_sbp(y_true_t, y_pred_t):
    return tf.reduce_mean(tf.abs(y_pred_t[:, 0] - y_true_t[:, 0]))

def mae_dbp(y_true_t, y_pred_t):
    return tf.reduce_mean(tf.abs(y_pred_t[:, 1] - y_true_t[:, 1]))


# GMU Layer
class GMULayer(layers.Layer):
    def __init__(self, hidden_dim, **kwargs):
        super().__init__(**kwargs)
        self.transform_x = layers.Dense(hidden_dim, activation="tanh")
        self.transform_y = layers.Dense(hidden_dim, activation="tanh")
        self.gate_layer = layers.Dense(hidden_dim, activation="sigmoid")

    def call(self, x, y):
        h_x = self.transform_x(x)
        h_y = self.transform_y(y)
        z = self.gate_layer(tf.concat([x, y], axis=-1))
        return z * h_x + (1.0 - z) * h_y
# GMU Layer for N modalities (generalized)

class GMULayerN(layers.Layer):
    def __init__(self, hidden_dim, num_modalities, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.num_modalities = num_modalities

        self.transforms = [
            layers.Dense(hidden_dim, activation="tanh")
            for _ in range(num_modalities)
        ]
        self.gate_layer = layers.Dense(hidden_dim * num_modalities)

    def call(self, inputs):
        # inputs: list/tuple of modality tensors
        # each tensor shape: (B, D_i)
        transformed = [
            transform(x) for transform, x in zip(self.transforms, inputs)
        ]  # list of (B, H)

        concat_inputs = tf.concat(inputs, axis=-1)  # (B, sum(D_i))
        gate_logits = self.gate_layer(concat_inputs)  # (B, M*H)
        gate_logits = tf.reshape(
            gate_logits, [-1, self.num_modalities, self.hidden_dim]
        )  # (B, M, H)

        gates = tf.nn.softmax(gate_logits, axis=1)  # (B, M, H)

        transformed = tf.stack(transformed, axis=1)  # (B, M, H)
        fused = tf.reduce_sum(gates * transformed, axis=1)  # (B, H)
        return fused

# Cross-Attention Layer
class CrossAttention(layers.Layer):
    def __init__(self, d_model, num_heads=4, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate,
        )
        self.dropout = layers.Dropout(dropout_rate)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)

        self.ffn = tf.keras.Sequential([
            layers.Dense(d_model * 4, activation="relu"),
            layers.Dense(d_model),
        ])
        self.dropout_ffn = layers.Dropout(dropout_rate)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, query, key, value, training=None):
        attn_out = self.mha(
            query=query,
            key=key,
            value=value,
            training=training,
        )
        attn_out = self.dropout(attn_out, training=training)
        out1 = self.norm1(query + attn_out)

        ffn_out = self.ffn(out1, training=training)
        ffn_out = self.dropout_ffn(ffn_out, training=training)
        out2 = self.norm2(out1 + ffn_out)
        return out2

*GMU is a vector-level fusion
*BCA is a sequence-level fusion
The proposed architecture is a multimodal deep learning framework that processes heterogeneous physiological signals through modality-specific encoders—namely, a CNN for modality 1 and BiLSTM networks for modalities 2 and 3—to capture complementary temporal and local patterns. The resulting sequence representations are projected into a shared latent space to ensure feature alignment across modalities. Fusion is then performed using one of two strategies: (i) a Gated Multimodal Unit (GMU), where temporally pooled modality representations are adaptively combined via learned gates, or (ii) bidirectional cross-attention (BCA), which models pairwise interactions between modalities at the sequence level before aggregating them into a unified representation. The fused features are subsequently passed through a fully connected regression head to jointly predict systolic and diastolic blood pressure.

In [ ]:
class AuxiliaryReliabilityFusion(layers.Layer):
    def __init__(self, hidden_dim, num_modalities, dominant_idx=0, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.num_modalities = num_modalities
        self.dominant_idx = dominant_idx
        self.num_aux = num_modalities - 1

        self.transforms = [
            layers.Dense(hidden_dim, activation="tanh")
            for _ in range(num_modalities)
        ]
        self.score_layers = [
            layers.Dense(hidden_dim)
            for _ in range(self.num_aux)
        ]
        self.reliability_layers = [
            layers.Dense(hidden_dim, activation="sigmoid")
            for _ in range(self.num_aux)
        ]

        self.concat = layers.Concatenate(axis=-1)
        self.softmax = layers.Softmax(axis=1)

    def call(self, inputs):
        transformed = [
            transform(x) for transform, x in zip(self.transforms, inputs)
        ]

        x_dom = inputs[self.dominant_idx]
        h_dom = transformed[self.dominant_idx]

        aux_hidden = []
        aux_scores = []
        aux_reliability = []

        aux_counter = 0
        for j, (x, h) in enumerate(zip(inputs, transformed)):
            if j == self.dominant_idx:
                continue

            pair = self.concat([x_dom, x, tf.abs(x_dom - x), x_dom * x])

            aux_hidden.append(h)
            aux_scores.append(self.score_layers[aux_counter](pair))
            aux_reliability.append(self.reliability_layers[aux_counter](pair))
            aux_counter += 1

        aux_hidden = tf.stack(aux_hidden, axis=1)
        aux_scores = tf.stack(aux_scores, axis=1)
        aux_reliability = tf.stack(aux_reliability, axis=1)

        alpha = self.softmax(aux_scores)
        comp = tf.reduce_sum(alpha * aux_reliability * aux_hidden, axis=1)

        fused = h_dom + comp
        return fused

In [ ]:

class DominantGuidedAuxiliaryGatedFusion(layers.Layer):
    def __init__(self, hidden_dim, num_modalities, dominant_idx=0, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.num_modalities = num_modalities
        self.dominant_idx = dominant_idx
        self.num_aux = num_modalities - 1

        if num_modalities < 2:
            raise ValueError("num_modalities must be at least 2")

        self.transforms = [
            layers.Dense(hidden_dim, activation="tanh", name=f"transform_{i}")
            for i in range(num_modalities)
        ]

        # Alpha: competition among auxiliary modalities
        self.score_layers = [
            layers.Dense(hidden_dim, name=f"score_{i}")
            for i in range(self.num_aux)
        ]

        # Joint interaction among auxiliary modalities only
        self.aux_joint_layer = layers.Dense(
            hidden_dim, activation="tanh", name="aux_joint"
        )

        # Merge weighted auxiliary evidence with joint auxiliary interaction
        self.comp_proj = layers.Dense(
            hidden_dim, activation="tanh", name="comp_proj"
        )

        # Beta: single gate for the whole complementary branch
        self.beta_layer = layers.Dense(
            hidden_dim, activation="sigmoid", name="beta_gate"
        )

        self.concat = layers.Concatenate(axis=-1)
        self.softmax = layers.Softmax(axis=1)

    def call(self, inputs):
        """
        inputs: list/tuple of pooled modality vectors
                e.g., [ppg, ecg, resp]
                each tensor shape: (B, D)

        Assumes all inputs are already pooled vectors.
        """
        if not isinstance(inputs, (list, tuple)):
            raise ValueError("inputs must be a list or tuple of modality tensors")

        if len(inputs) != self.num_modalities:
            raise ValueError(
                f"Expected {self.num_modalities} modalities, got {len(inputs)}"
            )

        # Shared hidden representations
        transformed = [
            transform(x) for transform, x in zip(self.transforms, inputs)
        ]  # list of (B, H)

        h_dom = transformed[self.dominant_idx]  # (B, H)

        aux_hidden_list = []
        aux_scores_list = []

        score_idx = 0
        for j, h in enumerate(transformed):
            if j == self.dominant_idx:
                continue

            # Dominant-conditioned pair representation for scoring
            pair = self.concat([
                h_dom,
                h,
                tf.abs(h_dom - h),
                h_dom * h
            ])  # (B, 4H)

            aux_hidden_list.append(h)
            aux_scores_list.append(self.score_layers[score_idx](pair))  # (B, H)
            score_idx += 1

        aux_hidden = tf.stack(aux_hidden_list, axis=1)   # (B, M-1, H)
        aux_scores = tf.stack(aux_scores_list, axis=1)   # (B, M-1, H)

        # Alpha: competition among auxiliary modalities
        alpha = self.softmax(aux_scores)                 # (B, M-1, H)

        # Weighted auxiliary evidence
        c_weighted = tf.reduce_sum(alpha * aux_hidden, axis=1)  # (B, H)

        # Joint auxiliary interaction using auxiliaries only
        joint_input = self.concat(aux_hidden_list)              # (B, (M-1)*H)
        c_joint = self.aux_joint_layer(joint_input)             # (B, H)

        # Complementary representation
        c = self.comp_proj(self.concat([c_weighted, c_joint]))  # (B, H)

        # Beta: should the complementary branch modify the dominant branch?
        beta = self.beta_layer(self.concat([h_dom, c]))         # (B, H)

        # Final fusion
        fused = h_dom + beta * c                                # (B, H)
        return fused

    def get_config(self):
        config = super().get_config()
        config.update({
            "hidden_dim": self.hidden_dim,
            "num_modalities": self.num_modalities,
            "dominant_idx": self.dominant_idx,
        })
        return config

In [ ]:
WINDOW_LEN = 3750
# ----------------------------
# Helper encoders
# ----------------------------
def cnn_encoder(inputs, dropout_rate=0.2, prefix="m1"):
    x = layers.Conv1D(32, 9, padding="same", name=f"{prefix}_conv1")(inputs)
    x = layers.BatchNormalization(name=f"{prefix}_bn1")(x)
    x = layers.Activation("relu", name=f"{prefix}_relu1")(x)
    x = layers.MaxPooling1D(pool_size=2, padding="same", name=f"{prefix}_pool1")(x)

    x = layers.Conv1D(64, 7, padding="same", name=f"{prefix}_conv2")(x)
    x = layers.BatchNormalization(name=f"{prefix}_bn2")(x)
    x = layers.Activation("relu", name=f"{prefix}_relu2")(x)
    x = layers.MaxPooling1D(pool_size=2, padding="same", name=f"{prefix}_pool2")(x)

    x = layers.Conv1D(128, 5, padding="same", name=f"{prefix}_conv3")(x)
    x = layers.BatchNormalization(name=f"{prefix}_bn3")(x)
    x = layers.Activation("relu", name=f"{prefix}_relu3")(x)
    x = layers.MaxPooling1D(pool_size=2, padding="same", name=f"{prefix}_pool3")(x)

    x = layers.Conv1D(256, 3, padding="same", name=f"{prefix}_conv4")(x)
    x = layers.BatchNormalization(name=f"{prefix}_bn4")(x)
    x = layers.Activation("relu", name=f"{prefix}_relu4")(x)
    return x




def bilstm_encoder(inputs, dropout_rate=0.2, prefix="m2"):
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True),
        name=f"{prefix}_bilstm1"
    )(inputs)
    x = layers.Dropout(dropout_rate, name=f"{prefix}_dropout")(x)

    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True),
        name=f"{prefix}_bilstm2"
    )(x)
    return x


# ----------------------------
# 2-modality model
# ----------------------------
def build_model_2modal(
    input_shape1=(WINDOW_LEN, 1),
    input_shape2=(WINDOW_LEN, 1),
    proj_dim=128,
    num_heads=4,
    dropout_rate=0.2,
    use_fusion='gmu',
):
    m1_input = layers.Input(shape=input_shape1, name="modality1_input")
    m2_input = layers.Input(shape=input_shape2, name="modality2_input")

    # Encoders
    x1 = cnn_encoder(m1_input, dropout_rate=dropout_rate, prefix="m1")
    x2 = bilstm_encoder(m2_input, dropout_rate=dropout_rate, prefix="m2")
    seq1 = layers.Dense(proj_dim, activation="relu", name="m1_seq_proj")(x1)
    seq1 = layers.LayerNormalization(name="m1_ln")(seq1)
    x2 = layers.Conv1D(proj_dim, kernel_size=3, strides=8, padding="same")(x2)
    seq2 = layers.Dense(proj_dim, activation="relu", name="m2_seq_proj")(x2)
    seq2 = layers.LayerNormalization(name="m2_ln")(seq2)

    # Fusion
    if use_fusion=="gmu":
        fused = GMULayer(proj_dim, name="gmu_fusion")(seq1, seq2)
        fused = layers.GlobalMaxPooling1D(name="gmu_gmp")(fused)
        fused = layers.Dense(256, activation="relu", name="gmu_dense")(fused)
        fused = layers.Dropout(dropout_rate, name="gmu_dropout")(fused)
    elif use_fusion=="aux":
        fused = DominantGuidedAuxiliaryGatedFusion(proj_dim,num_modalities=2,name="DominantGuidedAuxiliaryFusion")([seq1,seq2])
        fused = layers.GlobalMaxPooling1D(name="dominant_gmp")(fused)
    elif use_fusion=="bca":
        ca_12 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m1_to_m2"
        )(seq1, seq2, seq2)

        ca_21 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m2_to_m1"
        )(seq2, seq1, seq1)

        ca_12_vec = layers.GlobalMaxPooling1D(name="ca12_gmp")(ca_12)
        ca_21_vec = layers.GlobalMaxPooling1D(name="ca21_gmp")(ca_21)

        fused = layers.Concatenate(name="bca_concat")([ca_12_vec, ca_21_vec])
        fused = layers.Dense(256, activation="relu", name="fusion_dense")(fused)
        fused = layers.Dropout(dropout_rate, name="bca_fusion_dropout")(fused)

    elif use_fusion=='concat':
        fused = layers.Concatenate(name="concat_fusion")([seq1, seq2])
        fused = layers.GlobalMaxPooling1D(name="concat_gmp")(fused)

    # Head
    if use_fusion!='aux':
      fused = layers.Dense(128, activation="relu", name="fusion_dense_1")(fused)
    fused = layers.Dropout(dropout_rate, name="fusion_dropout")(fused)

    output = layers.Dense(
        2,
        activation="linear",
        dtype="float32",
        name="output"
    )(fused)

    model = Model(
        inputs=[m1_input, m2_input],
        outputs=output,
        name="multimodal_fusion_2modal"
    )
    return model


# ----------------------------
# 3-modality model
# Assumption:
#   m1 -> CNN backbone
#   m2 -> BiLSTM backbone
#   m3 -> BiLSTM backbone
# ----------------------------
def build_model_3modal(
    input_shape1=(WINDOW_LEN, 1),
    input_shape2=(WINDOW_LEN, 1),
    input_shape3=(WINDOW_LEN, 1),
    proj_dim=128,
    num_heads=4,
    dropout_rate=0.2,
    use_fusion='gmu',
):
    m1_input = layers.Input(shape=input_shape1, name="modality1_input")
    m2_input = layers.Input(shape=input_shape2, name="modality2_input")
    m3_input = layers.Input(shape=input_shape3, name="modality3_input")

    # Encoders
    x1 = cnn_encoder(m1_input, dropout_rate=dropout_rate, prefix="m1")
    seq1 = layers.Dense(proj_dim, activation="relu", name="m1_seq_proj")(x1)
    seq1 = layers.LayerNormalization(name="m1_ln")(seq1)

    x2 = bilstm_encoder(m2_input, dropout_rate=dropout_rate, prefix="m2")
    x2 = layers.Conv1D(proj_dim, kernel_size=3, strides=8, padding="same")(x2)
    seq2 = layers.Dense(proj_dim, activation="relu", name="m2_seq_proj")(x2)
    seq2 = layers.LayerNormalization(name="m2_ln")(seq2)


    x3 = bilstm_encoder(m3_input, dropout_rate=dropout_rate, prefix="m3")
    x3 = layers.Conv1D(proj_dim, kernel_size=3, strides=8, padding="same")(x3)
    seq3 = layers.Dense(proj_dim, activation="relu", name="m3_seq_proj")(x3)
    seq3 = layers.LayerNormalization(name="m3_ln")(seq3)

    # Fusion
    if use_fusion=='gmu':
        fused = GMULayerN(
            proj_dim,
            num_modalities=3,
            name="gmu_fusion_3modal"
        )([seq1, seq2, seq3])
        fused = layers.GlobalMaxPooling1D(name="gmu_gmp")(fused)
    elif use_fusion=='aux':
        fused = DominantGuidedAuxiliaryGatedFusion(proj_dim,num_modalities=3,name="DominantGuidedAuxiliaryFusion")([seq1,seq2,seq3])
        fused = layers.GlobalMaxPooling1D(name="dominant_gmp")(fused)

    elif use_fusion=='bca':
        ca_12 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m1_to_m2"
        )(seq1, seq2, seq2)
        ca_21 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m2_to_m1"
        )(seq2, seq1, seq1)

        ca_13 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m1_to_m3"
        )(seq1, seq3, seq3)
        ca_31 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m3_to_m1"
        )(seq3, seq1, seq1)

        ca_23 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m2_to_m3"
        )(seq2, seq3, seq3)
        ca_32 = CrossAttention(
            d_model=proj_dim,
            num_heads=num_heads,
            dropout_rate=dropout_rate,
            name="cross_attn_m3_to_m2"
        )(seq3, seq2, seq2)

        ca_12_vec = layers.GlobalMaxPooling1D(name="ca12_gmp")(ca_12)
        ca_21_vec = layers.GlobalMaxPooling1D(name="ca21_gmp")(ca_21)
        ca_13_vec = layers.GlobalMaxPooling1D(name="ca13_gmp")(ca_13)
        ca_31_vec = layers.GlobalMaxPooling1D(name="ca31_gmp")(ca_31)
        ca_23_vec = layers.GlobalMaxPooling1D(name="ca23_gmp")(ca_23)
        ca_32_vec = layers.GlobalMaxPooling1D(name="ca32_gmp")(ca_32)

        fused = layers.Concatenate(name="bca_concat")([
            ca_12_vec, ca_21_vec,
            ca_13_vec, ca_31_vec,
            ca_23_vec, ca_32_vec
        ])
        fused = layers.Dense(256, activation="relu", name="fusion_dense")(fused)
        fused = layers.Dropout(dropout_rate, name="bca_fusion_dropout")(fused)
    elif use_fusion=='concat':
        fused = layers.Concatenate(name="concat_fusion")([seq1, seq2, seq3])
        fused = layers.GlobalMaxPooling1D(name="concat_gmp")(fused)


    # Head
    if use_fusion!='aux':
      fused = layers.Dense(128, activation="relu", name="fusion_dense_1")(fused)
    fused = layers.Dropout(dropout_rate, name="fusion_dropout")(fused)

    output = layers.Dense(
        2,
        activation="linear",
        dtype="float32",
        name="output"
    )(fused)

    model = Model(
        inputs=[m1_input, m2_input, m3_input],
        outputs=output,
        name="multimodal_fusion_3modal"
    )
    return model


In [ ]:
# Build + Compile

model = build_model_3modal(
    input_shape1=(WINDOW_LEN, 1),
    input_shape2=(WINDOW_LEN, 1),
    input_shape3=(WINDOW_LEN, 1),
    use_fusion='aux'
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=3e-4,
    clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.Huber(delta=10.0),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae_all"),
        mae_sbp,
        mae_dbp,
        tf.keras.metrics.MeanSquaredError(name="mse_all"),
    ],
)

model.summary()
tf.keras.utils.plot_model(model, to_file="model.png", show_shapes=True)

In [ ]:
# CELL — Train

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
    verbose=1
)

print("Training finished. Best saved to:", best_path)

In [ ]:
# CELL — Evaluation

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# =========================
# Custom layer used in saved 3-modal model
# =========================
class GMULayerN(tf.keras.layers.Layer):
    def __init__(self, num_modalities, hidden_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_modalities = num_modalities
        self.hidden_dim = hidden_dim

        self.transforms = [
            tf.keras.layers.Dense(hidden_dim, activation="tanh")
            for _ in range(num_modalities)
        ]

        # IMPORTANT:
        # gate output = num_modalities * hidden_dim
        self.gate_layer = tf.keras.layers.Dense(
            num_modalities * hidden_dim,
            activation="sigmoid"
        )

    def call(self, inputs):
        # inputs = [x1, x2, x3], each shape (B, hidden_dim)
        transformed = [layer(x) for layer, x in zip(self.transforms, inputs)]

        concat_inputs = tf.concat(inputs, axis=-1)   # (B, num_modalities * hidden_dim)
        gates = self.gate_layer(concat_inputs)       # (B, num_modalities * hidden_dim)

        # reshape to (B, num_modalities, hidden_dim)
        gates = tf.reshape(gates, (-1, self.num_modalities, self.hidden_dim))

        fused = 0.0
        for i in range(self.num_modalities):
            gate_i = gates[:, i, :]                  # (B, hidden_dim)
            fused += gate_i * transformed[i]         # (B, hidden_dim)

        return fused

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_modalities": self.num_modalities,
            "hidden_dim": self.hidden_dim,
        })
        return config


# =========================
# Metrics
# =========================
def mae_sbp(y_true_t, y_pred_t):
    return tf.reduce_mean(tf.abs(y_pred_t[:, 0] - y_true_t[:, 0]))

def mae_dbp(y_true_t, y_pred_t):
    return tf.reduce_mean(tf.abs(y_pred_t[:, 1] - y_true_t[:, 1]))


# =========================
# Load best model
# =========================
best_model = tf.keras.models.load_model(
    best_path,
    custom_objects={
        "GMULayerN": GMULayerN,
        "mae_sbp": mae_sbp,
        "mae_dbp": mae_dbp,
    },
    compile=False
)

# Predict on test dataset
y_pred = best_model.predict(test_data, verbose=0)

# Ground truth
y_true = []
for _, yb in test_data:
    y_true.append(yb.numpy())
y_true = np.concatenate(y_true, axis=0)

# Metrics
model_metrics = {
    "name": run_name,
    "mae_sbp": float(np.mean(np.abs(y_pred[:, 0] - y_true[:, 0]))),
    "mae_dbp": float(np.mean(np.abs(y_pred[:, 1] - y_true[:, 1]))),
    "mae_all": float(np.mean(np.abs(y_pred - y_true))),
    "mse_sbp": float(np.mean((y_pred[:, 0] - y_true[:, 0]) ** 2)),
    "mse_dbp": float(np.mean((y_pred[:, 1] - y_true[:, 1]) ** 2)),
    "mse_all": float(np.mean((y_pred - y_true) ** 2)),
    "best_model_path": best_path,
}

print(f"=== {run_name} ===")
print("MAE ALL:", model_metrics["mae_all"])
print("MAE SBP:", model_metrics["mae_sbp"])
print("MAE DBP:", model_metrics["mae_dbp"])
print("MSE ALL:", model_metrics["mse_all"])
print("MSE SBP:", model_metrics["mse_sbp"])
print("MSE DBP:", model_metrics["mse_dbp"])

In [ ]:
# CELL 14 — Save results

import json
import pandas as pd
import os

mean_c = [mean1, mean2]
std_c  = [std1, std2]

results = {
    "run_name": run_name,
    "modalities": [MOD1, MOD2],
    "model_type": MODEL_TYPE,
    "window_len": int(WINDOW_LEN),
    "batch": int(BATCH),
    "epochs": int(EPOCHS),
    "steps_per_epoch": int(STEPS_PER_EPOCH),
    "val_steps": int(VAL_STEPS),
    "train_zscore_mean_per_channel": [float(x) for x in mean_c],
    "train_zscore_std_per_channel":  [float(x) for x in std_c],
    "baseline": baseline,
    "model": model_metrics,
    "dataset_stats": dataset_stats,
}

json_path = os.path.join(ckpt_dir, f"results_{run_name}.json")
csv_path  = os.path.join(ckpt_dir, f"results_{run_name}.csv")

with open(json_path, "w") as f:
    json.dump(results, f, indent=2)

row = {
    "run_name": run_name,
    "mod1": MOD1,
    "mod2": MOD2,
    "model_type": MODEL_TYPE,
    "window_len": int(WINDOW_LEN),
    "batch": int(BATCH),

    "mean_c0": float(mean_c[0]),
    "std_c0":  float(std_c[0]),
    "mean_c1": float(mean_c[1]),
    "std_c1":  float(std_c[1]),

    "baseline_mae_all": baseline["mae_all"],
    "baseline_mae_sbp": baseline["mae_sbp"],
    "baseline_mae_dbp": baseline["mae_dbp"],
    "baseline_mse_all": baseline["mse_all"],
    "baseline_mse_sbp": baseline["mse_sbp"],
    "baseline_mse_dbp": baseline["mse_dbp"],

    "model_mae_all": model_metrics["mae_all"],
    "model_mae_sbp": model_metrics["mae_sbp"],
    "model_mae_dbp": model_metrics["mae_dbp"],
    "model_mse_all": model_metrics["mse_all"],
    "model_mse_sbp": model_metrics["mse_sbp"],
    "model_mse_dbp": model_metrics["mse_dbp"],

"best_model_path": model_metrics.get("best_model_path", None),}

pd.DataFrame([row]).to_csv(csv_path, index=False)

print(" Saved JSON:", json_path)
print(" Saved CSV :", csv_path)

In [ ]:
# CELL 15 — Save plots

import os
import numpy as np
import matplotlib.pyplot as plt

plots_dir = os.path.join(ckpt_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

# Scatter SBP
plt.figure(figsize=(6,5))
plt.scatter(y_true[:, 0], y_pred[:, 0], s=6)
plt.xlabel("True SBP"); plt.ylabel("Pred SBP"); plt.title("SBP: True vs Pred")
plt.grid(True)
p1 = os.path.join(plots_dir, f"{run_name}_SBP_scatter.png")
plt.savefig(p1, dpi=200, bbox_inches="tight")
plt.show()

# Scatter DBP
plt.figure(figsize=(6,5))
plt.scatter(y_true[:, 1], y_pred[:, 1], s=6)
plt.xlabel("True DBP"); plt.ylabel("Pred DBP"); plt.title("DBP: True vs Pred")
plt.grid(True)
p2 = os.path.join(plots_dir, f"{run_name}_DBP_scatter.png")
plt.savefig(p2, dpi=200, bbox_inches="tight")
plt.show()

# Residual hist
err_sbp = y_pred[:, 0] - y_true[:, 0]
err_dbp = y_pred[:, 1] - y_true[:, 1]

plt.figure(figsize=(7,4))
plt.hist(err_sbp, bins=50)
plt.title("SBP Residuals (Pred-True)"); plt.xlabel("mmHg"); plt.ylabel("count")
p3 = os.path.join(plots_dir, f"{run_name}_SBP_residual_hist.png")
plt.savefig(p3, dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,4))
plt.hist(err_dbp, bins=50)
plt.title("DBP Residuals (Pred-True)"); plt.xlabel("mmHg"); plt.ylabel("count")
p4 = os.path.join(plots_dir, f"{run_name}_DBP_residual_hist.png")
plt.savefig(p4, dpi=200, bbox_inches="tight")
plt.show()

# Training curve
if "history" in globals() and hasattr(history, "history"):
    hist = history.history
    plt.figure(figsize=(7,4))
    plt.plot(hist.get("loss", []), label="train_loss")
    plt.plot(hist.get("val_loss", []), label="val_loss")
    plt.title("Training Curve: Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.grid(True); plt.legend()
    p5 = os.path.join(plots_dir, f"{run_name}_loss_curve.png")
    plt.savefig(p5, dpi=200, bbox_inches="tight")
    plt.show()
    print(" Saved:", p5)

print(" Saved plots to:", plots_dir)
print(" -", p1)
print(" -", p2)
print(" -", p3)
print(" -", p4)
